# 03 Risk Classification

This notebook demonstrates the end-to-end risk classification workflow:

- Load processed feature arrays when available
- Train baseline and core models
- Tune the best model with time-series cross-validation
- Evaluate on the test set
- Save the best model in joblib format


In [ ]:
from pathlib import Path
import json
import sys

import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
SRC_DIR = PROJECT_ROOT / 'src'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
MODEL_DIR = PROJECT_ROOT / 'src' / 'models'
FIGURE_DIR = PROJECT_ROOT / 'results' / 'figures'

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from features import CreditDataPreprocessor, CreditFeatureSelector, CreditSampler
from models import CreditModelEvaluator, CreditRiskClassifier


In [ ]:
# Configuration
TARGET_COLUMN = 'preloan_risk_label'
USE_SUBSET_FOR_DEMO = True
TRAIN_ROWS = 120000
VAL_ROWS = 30000
TEST_ROWS = 30000

# Set USE_SUBSET_FOR_DEMO to False when you want to use the full dataset.
print({
    'TARGET_COLUMN': TARGET_COLUMN,
    'USE_SUBSET_FOR_DEMO': USE_SUBSET_FOR_DEMO,
    'TRAIN_ROWS': TRAIN_ROWS,
    'VAL_ROWS': VAL_ROWS,
    'TEST_ROWS': TEST_ROWS,
})


In [ ]:
# Helper functions
def build_dataframe_from_npy(array_path: Path, columns: list[str]) -> pd.DataFrame:
    array = np.load(array_path)
    return pd.DataFrame(array, columns=columns)

def maybe_subset_frame(frame: pd.DataFrame, row_limit: int) -> pd.DataFrame:
    if not USE_SUBSET_FOR_DEMO:
        return frame
    return frame.iloc[: min(row_limit, len(frame))].copy()

def maybe_subset_series(series: pd.Series, row_limit: int) -> pd.Series:
    if not USE_SUBSET_FOR_DEMO:
        return series
    return series.iloc[: min(row_limit, len(series))].copy()

def resolve_split_path(split_name: str) -> Path:
    purified_path = PROCESSED_DIR / f'purified_{split_name}.csv'
    return purified_path if purified_path.exists() else PROCESSED_DIR / f'{split_name}.csv'

def prepare_feature_sets():
    manifest_path = PROCESSED_DIR / 'feature_manifest.json'
    train_resampled_path = PROCESSED_DIR / 'X_train_resampled.npy'
    y_train_resampled_path = PROCESSED_DIR / 'y_train_resampled.npy'
    val_path = PROCESSED_DIR / 'X_val.npy'
    y_val_path = PROCESSED_DIR / 'y_val.npy'
    test_path = PROCESSED_DIR / 'X_test.npy'
    y_test_path = PROCESSED_DIR / 'y_test.npy'

    arrays_available = all([
        manifest_path.exists(),
        train_resampled_path.exists(),
        y_train_resampled_path.exists(),
        val_path.exists(),
        y_val_path.exists(),
        test_path.exists(),
        y_test_path.exists(),
    ])

    if arrays_available:
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
        feature_names = manifest['selected_feature_names']
        X_train_resampled = build_dataframe_from_npy(train_resampled_path, feature_names)
        y_train_resampled = pd.Series(np.load(y_train_resampled_path), name=TARGET_COLUMN).astype(int)
        X_val = build_dataframe_from_npy(val_path, feature_names)
        y_val = pd.Series(np.load(y_val_path), name=TARGET_COLUMN).astype(int)
        X_test = build_dataframe_from_npy(test_path, feature_names)
        y_test = pd.Series(np.load(y_test_path), name=TARGET_COLUMN).astype(int)
    else:
        X_train_resampled = y_train_resampled = X_val = y_val = X_test = y_test = None

    train_df = pd.read_csv(PROCESSED_DIR / 'train.csv', low_memory=False)
    val_df = pd.read_csv(PROCESSED_DIR / 'val.csv', low_memory=False)
    test_df = pd.read_csv(PROCESSED_DIR / 'test.csv', low_memory=False)

    train_df = maybe_subset_frame(train_df, TRAIN_ROWS)
    val_df = maybe_subset_frame(val_df, VAL_ROWS)
    test_df = maybe_subset_frame(test_df, TEST_ROWS)

    y_train = train_df[TARGET_COLUMN].astype(int).copy()
    y_val_raw = val_df[TARGET_COLUMN].astype(int).copy()
    y_test_raw = test_df[TARGET_COLUMN].astype(int).copy()

    preprocessor = CreditDataPreprocessor(target_column=TARGET_COLUMN)
    X_train_processed = preprocessor.fit_transform(train_df)
    X_val_processed = preprocessor.transform(val_df)
    X_test_processed = preprocessor.transform(test_df)

    selector = CreditFeatureSelector(
        variance_threshold=0.01,
        correlation_threshold=0.8,
        top_k_features=50,
        use_pca=False,
    )
    X_train_selected = selector.fit_transform(X_train_processed, y_train)
    X_val_selected = selector.transform(X_val_processed)
    X_test_selected = selector.transform(X_test_processed)

    if X_train_resampled is None or USE_SUBSET_FOR_DEMO:
        sampler = CreditSampler(random_state=42)
        X_train_resampled, y_train_resampled = sampler.fit_resample(X_train_selected, y_train)
        X_val = X_val_selected.copy()
        y_val = y_val_raw.copy()
        X_test = X_test_selected.copy()
        y_test = y_test_raw.copy()
    else:
        X_train_resampled = maybe_subset_frame(X_train_resampled, TRAIN_ROWS)
        y_train_resampled = maybe_subset_series(y_train_resampled, TRAIN_ROWS)
        X_val = maybe_subset_frame(X_val, VAL_ROWS)
        y_val = maybe_subset_series(y_val, VAL_ROWS)
        X_test = maybe_subset_frame(X_test, TEST_ROWS)
        y_test = maybe_subset_series(y_test, TEST_ROWS)

    return {
        'X_train_selected': X_train_selected,
        'y_train': y_train,
        'X_val_selected': X_val_selected,
        'y_val': y_val_raw,
        'X_test_selected': X_test_selected,
        'y_test': y_test_raw,
        'X_train_resampled': X_train_resampled,
        'y_train_resampled': y_train_resampled,
        'X_val_final': X_val,
        'y_val_final': y_val,
        'X_test_final': X_test,
        'y_test_final': y_test,
    }


In [ ]:
# Load or regenerate feature sets
artifacts = prepare_feature_sets()

X_train_selected = artifacts['X_train_selected']
y_train = artifacts['y_train']
X_val_selected = artifacts['X_val_selected']
y_val = artifacts['y_val']
X_test_selected = artifacts['X_test_selected']
y_test = artifacts['y_test']
X_train_resampled = artifacts['X_train_resampled']
y_train_resampled = artifacts['y_train_resampled']
X_val_final = artifacts['X_val_final']
y_val_final = artifacts['y_val_final']
X_test_final = artifacts['X_test_final']
y_test_final = artifacts['y_test_final']

print('X_train_selected:', X_train_selected.shape)
print('X_train_resampled:', X_train_resampled.shape)
print('X_val_final:', X_val_final.shape)
print('X_test_final:', X_test_final.shape)


In [ ]:
# Train baseline and core models
model_types = [
    'logistic_regression',
    'decision_tree',
    'random_forest',
    'xgboost',
    'lightgbm',
]

model_results = []
trained_models = {}

for model_type in model_types:
    classifier = CreditRiskClassifier(model_type=model_type)
    classifier.fit(X_train_resampled, y_train_resampled)
    val_pred = classifier.predict(X_val_final)
    result = {
        'model_type': model_type,
        'val_accuracy': accuracy_score(y_val_final, val_pred),
        'val_f1_weighted': f1_score(y_val_final, val_pred, average='weighted', zero_division=0),
    }
    model_results.append(result)
    trained_models[model_type] = classifier

model_result_df = pd.DataFrame(model_results).sort_values('val_f1_weighted', ascending=False)
display(model_result_df)

best_model_type = model_result_df.iloc[0]['model_type']
print('Best validation model:', best_model_type)


In [ ]:
# Hyperparameter tuning on the best model using time-series CV
best_classifier = CreditRiskClassifier(model_type=best_model_type)
best_classifier.hyperparameter_tune(
    X_train_selected,
    y_train,
    X_val_selected,
    y_val,
)

tuning_summary = best_classifier.get_tuning_summary()
display(pd.DataFrame([tuning_summary.__dict__]))


In [ ]:
# Final test evaluation
evaluator = CreditModelEvaluator(
    model=best_classifier,
    X_test=X_test_selected,
    y_test=y_test,
)
evaluation_results = evaluator.evaluate_classification()
plot_paths = evaluator.plot_evaluation_results()
explanation_results = evaluator.explain_model(X_test_selected)

print('Evaluation results:')
print(json.dumps(evaluation_results, indent=2, ensure_ascii=False))
print('Plot paths:')
print(json.dumps(plot_paths, indent=2, ensure_ascii=False))
print('Explainability outputs:')
print(json.dumps(explanation_results, indent=2, ensure_ascii=False))


In [ ]:
# Save the best tuned model and evaluation outputs
model_output_path = MODEL_DIR / f'best_{best_model_type}_risk_classifier.joblib'
joblib.dump(best_classifier.best_model_, model_output_path)

metrics_output_path = MODEL_DIR / 'best_model_evaluation.json'
metrics_output_path.write_text(
    json.dumps(evaluation_results, indent=2, ensure_ascii=False),
    encoding='utf-8',
)

print('Saved model to:', model_output_path)
print('Saved evaluation metrics to:', metrics_output_path)
